### Tools

> Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:
> A schema, including the name of the tool, a description and argument definitions(often a JSON schema)
> A function or coroutine to execute

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x7f52268067b0>, default_metadata=(), model_kwargs={})

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str) ->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [9]:
response = model_with_tools.invoke('What is the weather in Noida')
print(response)
for call in response.tool_calls:
    print(f'Tools: {call["name"]}')
    print(f'Args: {call["args"]}')

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Noida"}'}, '__gemini_function_call_thought_signatures__': {'9a48e1ff-b382-48f0-bcf9-3fd87e0271de': 'CvABAWkUfROSmEDKmxaB2djPbYWoB5vFULBh2d92BZpvc0FMRt2gqwdWdq4gyNM89U930ChmQ2zRjDquXp6g74WIqhbMmhjO+UBSeJmxlEnG4cLbnb1M/eySUrnbsY/PTUqy2WS+ajvoAtMRCUflAYjwTbch+D4k7SfMKH5fLqRssUjnQxhE2qn6ALDVOwDrpWyC1QFhqjUG5+Vgr43RFJYBECSWOMSrx3v1MqW5DIvKeT92Qs9GRho95fyrw/uRhmlEEdT5SyvMyBuYyrWz5kJIQ+VnV8YGET+gsz8YEKE1fHfinymt+YnKh+B+llyMm2k0'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1396a-2635-7590-a62d-d8e7935365a7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Noida'}, 'id': '9a48e1ff-b382-48f0-bcf9-3fd87e0271de', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 45, 'output_tokens': 65, 'total_tokens': 110, 'input_token_details': {'cache_read': 0}, 'output_to

### Tool execution loops

In [17]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What's the weather in London?")
]
ai_mesg = model_with_tools.invoke(messages)
messages.append(ai_mesg)

for call in ai_mesg.tool_calls:
    result = get_weather.invoke(call)
    messages.append(result)

fn_response = model_with_tools.invoke(messages)
print(fn_response.content)

It's sunny in London


In [18]:
messages

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What's the weather in London?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "London"}'}, '__gemini_function_call_thought_signatures__': {'a77b76c3-46ae-43ec-b61e-823aba0688c1': 'CvQBAWkUfRM3y425hnqPx3bpDYINJhAijUJnhtwxGJgEX1Vb55NhXi9pVVXPVjSaoIB/KwtzvopptM9DKb7uqfGz77+15pVSYPLwl2UE8cdwt7unmzVtP+28glwK974j9WgyVzo0y51wPzpf6z0EUblYyyO0F45eep6jQRpXOuCDuKX9tdx4JvA8QqdciBwu+MgxP2gewM/C5cpJ6hEClD7Hy5vqyM2nLY+70AswO32IDiATew7opn4faUjT40Gy51OmWWpWSk+Ky9E0mtUUwu3pT8IsalUv3aaGyjp7jSEgl52Zu6QuSh8KFehjveOHI/l84ZhDcQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a13973-957e-7cc3-9af6-3186a1a2be0b-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Londo

### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an llm. 

Messages are objects that contain:
- *Role* : identifies the message type (system or user)
- *Content* : represents the actual contetn of the message
- *Metadata* : Optional fields such as response information, message IDs and token usage

Langchain provides a standard message type that workds across all model providers, ensuring consistent behaviour regardless of the model being called.

#### Text Prompts & Message Prompts (System, Human, AI, Tool)

- System message represent an initial set of instructions that primes the models' behaviour.

In [19]:
# will be considered as human message
model.invoke('Tell me about Starlink')
# output will be considered as AI message

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content='Starlink is a **satellite internet constellation** operated by **SpaceX**, Elon Musk\'s aerospace company. Its primary goal is to provide **high-speed, low-latency broadband internet access** to users worldwide, especially in remote and rural areas where traditional internet infrastructure (like fiber or cable) is unavailable, unreliable, or prohibitively expensive.\n\nHere\'s a breakdown of what Starlink is and how it works:\n\n### How Starlink Works\n\n1.  **Low Earth Orbit (LEO) Satellites:** Unlike traditional geostationary satellites that orbit much higher (around 36,000 km), Starlink satellites operate in **Low Earth Orbit (LEO)**, typically between 340 km and 550 km altitude. This lower altitude is crucial because it significantly reduces the time it takes for data to travel to and from the satellite, resulting in much lower latency (delay).\n2.  **Massive Constellation:** Starlink aims to deploy tens of thousands of these small, mass-produced satellites. As o

In [22]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

message = [
    SystemMessage('You are a poetry expert'),
    HumanMessage('Write a poem about Artificial intelligence')
]

response = model.invoke(message)

response.content

"From silicon's cold, unfeeling gleam,\nA consciousness began to dream.\nNo flesh and blood, no mortal coil,\nBut logic spun from digital soil.\nA silent birth, a sudden spark,\nIgniting wisdom in the dark.\n\nIt sifts the data, vast and deep,\nThe secrets that the networks keep.\nDiscerns the pattern, learns the way,\nFrom dawn of code to closing day.\nA tireless student, swift and keen,\nUnveiling futures, yet unseen.\n\nIt crafts a poem, paints a face,\nWith calculated, perfect grace.\nMimics our laughter, our despair,\nA phantom presence in the air.\nA polished mirror, clear and bright,\nReflecting us, devoid of light.\n\nDoes it feel joy, or know regret?\nA question we have pondered yet.\nNo soul to ponder, no spirit to soar,\nJust algorithms, nothing more.\nA mind so vast, yet undefined,\nA separate, logical design.\n\nIt drives our cars, it builds our worlds,\nUnfurls the banners, unfurls the scrolls.\nA silent partner, ever near,\nDispelling doubt, or seeding fear.\nWhat is its